## Prerequisites

This notebook demonstrates how to invoke a **deployed real-time inference endpoint** on Azure for the Social Determinants of Health (SDOH) model. Follow the steps below in order before running this notebook.

### Deployment Sequence

1. **Acquire the Azure Application Offer** — Subscribe to the *extract_social_determinants_health* offer in the [Azure Marketplace](https://marketplace.azure.com) to obtain the Solution Template.
2. **Launch guided deployment** — From the Marketplace listing, start the guided deployment flow for the solution.
3. **Supply configuration** — Complete the guided UX forms (defined by the offer's `createUiDefinition`) with the required parameters.
4. **Deploy resources** — Confirm deployment; Azure provisions the resources defined in the offer's `mainTemplate.json`.
5. **Wait for deployment to succeed** — Verify the deployment reaches a *Succeeded* state before proceeding. Note the endpoint name from the deployment output.
6. **Run this notebook** — Execute the cells below against the real-time inference endpoint provisioned by the deployment.

### Environment

- **Notebook runtime**: This notebook is intended to run in **Azure Machine Learning workspace notebooks** (e.g. via an Azure ML Studio compute instance or Studio notebooks).
- **Jupyter kernel**: Use a kernel with the `azure.ai.ml` Python package installed, for example **Python 3.10 — SDK v2** (`python310-sdkv2`).
- **Authentication**: The kernel must be authenticated to Azure so that Azure ML SDK calls succeed (e.g. `DefaultAzureCredential()`). The authenticated identity must be granted Azure RBAC permissions appropriate for reading the workspace and online endpoint configuration, and for retrieving endpoint credentials. Consult your organization's access-policy guidance and the [Azure ML endpoint credential documentation](https://learn.microsoft.com/azure/machine-learning/how-to-manage-endpoints) for details.
- **Network access**: The notebook's compute instance must be able to reach the endpoint's `scoring_uri` (no outbound firewall or VNET restrictions that block it).

## Explain Clinical Document - Social Determinants of Health (SDOH)-Small

- **Model**: [explain_clinical_doc_sdoh_small_en](https://nlp.johnsnowlabs.com/2024/09/27/explain_clinical_doc_sdoh_small_en.html)
- **Model Description**: This pipeline extracts clinical/medical entities related to Social Determinants of Health (SDOH) from text, assigning assertion statuses to the extracted entities, and establishing relations between the extracted entities.

In [ ]:
ENDPOINT_NAME = "<CUSTOMER_TO_PUT_DEPLOYED_ENDPOINT_NAME>"

### Input Format

To use the model, you need to provide input in one of the following supported formats:

#### JSON Format

Provide input as JSON. We support two variations within this format:

1. **Array of Text Documents**: 
   Use an array containing multiple text documents. Each element represents a separate text document.

   ```json
   {
       "text": [
           "Text document 1",
           "Text document 2",
           ...
       ]
   }

    ```

2. **Single Text Document**:
   Provide a single text document as a string.


   ```json
    {
        "text": "Single text document"
    }
   ```

#### JSON Lines (JSONL) Format

Provide input in JSON Lines format, where each line is a JSON object representing a text document.

```
{"text": "Text document 1"}
{"text": "Text document 2"}
```

> **Note:** Demonstration inputs used with this endpoint should consist of approved synthetic or properly de-identified data. Never submit real protected health information (PHI) without explicit authorization and the required safeguarding controls.

### Authentication and Endpoint Lookup

The cell below uses `DefaultAzureCredential()` to authenticate to Azure, then retrieves the endpoint's scoring URI and primary access key from the workspace configuration. If you see authentication errors, verify that you are logged into Azure Cloud Shell / your compute instance and that your identity has sufficient role assignments on the workspace.

In [3]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import requests
import json
import pandas as pd


ml_client = MLClient.from_config(DefaultAzureCredential())
endpoint_url = ml_client.online_endpoints.get(ENDPOINT_NAME).scoring_uri
key = ml_client.online_endpoints.get_keys(ENDPOINT_NAME).primary_key


def invoke_realtime_endpoint(record, content_type="application/json", accept="application/json"):
    response = requests.post(
        endpoint_url,
        headers={
            "Authorization": f"Bearer {key}",
            "Content-Type": content_type,
        },
        data=json.dumps(record, ensure_ascii=False) if content_type == "application/json" else record,
      
    )

    response_body = response.content.decode("utf-8")

    if accept == "application/json":
        return json.loads(response_body)
    elif accept == "application/jsonlines":
        return [json.loads(line) for line in response_body.strip().split('\n') if line]
    else:
        raise ValueError(f"Unsupported accept type: {accept}")

Found the config file in: ./config.json


### Sample Data (Synthetic — for demonstration only)

The following cells define sample clinical text documents used in the examples below. When using this notebook with your own data, ensure all inputs are approved synthetic or properly de-identified. Never submit real protected health information (PHI) without explicit authorization and appropriate controls.

In [4]:
docs = [
    "Kate a cleaning assistant without health insurance or paid sick leave, has struggled with alcohol abuse since her teens, drinking beer daily for 30 years and smoking a pack of cigarettes daily. With a DUI in April and a recent court date, she leans on her Catholic faith for support, alongside managing depression and considering rehab.",
    "Jane works as a cleaning assistant and cannot access health insurance or paid sick leave. She has a son, a student at college. Pt with likely long-standing depression. She is aware she needs rehab. Pt reports having her catholic faith as a means of support as well.  She has a long history of etoh abuse, beginning in her teens. She reports she has been a daily drinker for 30 years, most recently drinking beer daily. She smokes a pack of cigarettes a day. She had DUI in April and was due to court this week."
]

sample_text = """The patient reported experiencing symptoms of anxiety and depression, which have been affecting his quality of life. 
He reported a history of childhood trauma related to violence and abuse in his household, which has contributed to his smoking, alcohol use and current mental health struggles. 
He denied any recent substance use or sexual activity and reported being monogamous in his relationship with his wife. 
The patient is an immigrant and speaks English as a second language. 
He reported difficulty accessing healthcare due to lack of medical insurance. 
He has a herniated disc, hypertension, coronary artery disease (CAD) and diabetes mellitus. 
The patient has a manic disorder, is presently psychotic and shows impulsive behavior. He has been disabled since 2001."""

### JSON


In [5]:
input_json_data = {"text": sample_text}
data = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
data["predictions"]

{'ner_predictions': [{'ner_chunk': 'anxiety',
   'begin': 46,
   'end': 52,
   'ner_label': 'Mental_Health',
   'ner_confidence': '0.9897'},
  {'ner_chunk': 'depression',
   'begin': 58,
   'end': 67,
   'ner_label': 'Mental_Health',
   'ner_confidence': '0.9938'},
  {'ner_chunk': 'his',
   'begin': 96,
   'end': 98,
   'ner_label': 'Gender',
   'ner_confidence': '0.992'},
  {'ner_chunk': 'quality of life',
   'begin': 100,
   'end': 114,
   'ner_label': 'Quality_Of_Life',
   'ner_confidence': '0.6252'},
  {'ner_chunk': 'He',
   'begin': 118,
   'end': 119,
   'ner_label': 'Gender',
   'ner_confidence': '0.9996'},
  {'ner_chunk': 'childhood trauma',
   'begin': 143,
   'end': 158,
   'ner_label': 'Chidhood_Event',
   'ner_confidence': '0.74660003'},
  {'ner_chunk': 'violence',
   'begin': 171,
   'end': 178,
   'ner_label': 'Violence_Or_Abuse',
   'ner_confidence': '0.5394'},
  {'ner_chunk': 'abuse',
   'begin': 184,
   'end': 188,
   'ner_label': 'Violence_Or_Abuse',
   'ner_confidenc

### JSON Lines


In [6]:
def create_jsonl(records):
    if isinstance(records, str):
        records = [records]
    json_records = [{"text": text} for text in records]
    json_lines = "\n".join(json.dumps(record) for record in json_records)
    return json_lines

In [8]:
input_jsonl_data = create_jsonl(docs)
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
[print(prediction) for prediction in data]

{'predictions': {'ner_predictions': [{'ner_chunk': 'cleaning assistant', 'begin': 7, 'end': 24, 'ner_label': 'Employment', 'ner_confidence': '0.6665'}, {'ner_chunk': 'health insurance', 'begin': 34, 'end': 49, 'ner_label': 'Insurance_Status', 'ner_confidence': '0.4379'}, {'ner_chunk': 'alcohol abuse', 'begin': 90, 'end': 102, 'ner_label': 'Alcohol', 'ner_confidence': '0.6923'}, {'ner_chunk': 'her', 'begin': 110, 'end': 112, 'ner_label': 'Gender', 'ner_confidence': '0.9923'}, {'ner_chunk': 'teens', 'begin': 114, 'end': 118, 'ner_label': 'Age', 'ner_confidence': '0.7475'}, {'ner_chunk': 'drinking', 'begin': 121, 'end': 128, 'ner_label': 'Alcohol', 'ner_confidence': '0.9493'}, {'ner_chunk': 'beer', 'begin': 130, 'end': 133, 'ner_label': 'Alcohol', 'ner_confidence': '0.9728'}, {'ner_chunk': 'daily', 'begin': 135, 'end': 139, 'ner_label': 'Substance_Frequency', 'ner_confidence': '0.9908'}, {'ner_chunk': '30 years', 'begin': 145, 'end': 152, 'ner_label': 'Substance_Duration', 'ner_confidence

[None, None]